# Agentic Flink — Quickstart Notebook

Hands-on tour of the framework via the Python facade (JPype-backed).
Runs top-to-bottom; each section is self-contained.

Sections:
1. JVM bootstrap + framework discovery
2. Web fetch via `WebFetchTool`
3. Document extraction (Jsoup + Tika)
4. Recursive text chunking
5. In-JVM vector memory (brute-force + HNSW)
6. Agent + Python `@tool` (no LLM call required to inspect)
7. Optional: live agent run against Ollama (skipped if not reachable)

**Prereqs:** `pip install -e python/`, framework jar built (`mvn -DskipTests package`),
`AGENTIC_FLINK_JAR` env var pointing at `target/agentic-flink-*.jar`.

## 1. Bootstrap

In [1]:
import os, sys, pathlib, subprocess

repo_root = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == 'notebooks' else pathlib.Path.cwd()

# Resolve the framework jar.
if 'AGENTIC_FLINK_JAR' not in os.environ:
    jars = sorted((repo_root / 'target').glob('agentic-flink-*.jar'))
    jars = [j for j in jars if 'original' not in j.name and 'sources' not in j.name]
    assert jars, f'No agentic-flink jar found under {repo_root}/target — run mvn -DskipTests package'
    os.environ['AGENTIC_FLINK_JAR'] = str(jars[-1])

# Resolve the rest of the runtime classpath (Flink, LangChain4J, Jsoup, ...).
# Cache it so this is a one-shot cost per notebook session.
cp_cache = repo_root / 'target' / 'runtime-classpath.txt'
if not cp_cache.exists():
    print('Resolving runtime classpath via Maven (one-off, ~10s)...')
    subprocess.run(
        ['mvn', '-q', 'dependency:build-classpath',
         f'-Dmdep.outputFile={cp_cache}',
         '-Dmdep.includeScope=runtime'],
        cwd=repo_root, check=True,
    )

extra_jars = [j for j in cp_cache.read_text().strip().split(':') if j]
print(f'Framework jar : {os.environ["AGENTIC_FLINK_JAR"]}')
print(f'Extra jars    : {len(extra_jars)} (Flink, LangChain4J, Jsoup, Tika, ...)')

import agentic_flink as af
af.start_jvm(extra_jars=extra_jars)
print(f'JVM started   : {af.is_started()}')

Framework jar : /Users/bengamble/Agentic-Flink/target/agentic-flink-1.0.0-SNAPSHOT.jar
Extra jars    : 237 (Flink, LangChain4J, Jsoup, Tika, ...)


JVM started   : True


In [2]:
# Confirm key framework classes resolve.
from agentic_flink import jclass

for fqcn in [
    'org.agentic.flink.dsl.Agent',
    'org.agentic.flink.web.WebFetchTool',
    'org.agentic.flink.ingest.RecursiveTextChunker',
    'org.agentic.flink.memory.vector.FlinkStateVectorMemory',
    'org.agentic.flink.memory.vector.FlinkStateHnswVectorMemory',
    'org.agentic.flink.corpus.SingleOperatorCorpus',
    'org.agentic.flink.tool.ToolRegistry',
]:
    jclass(fqcn)  # raises if not on the classpath
    print(f'  ✓ {fqcn}')

  ✓ org.agentic.flink.dsl.Agent
  ✓ org.agentic.flink.web.WebFetchTool
  ✓ org.agentic.flink.ingest.RecursiveTextChunker
  ✓ org.agentic.flink.memory.vector.FlinkStateVectorMemory
  ✓ org.agentic.flink.memory.vector.FlinkStateHnswVectorMemory
  ✓ org.agentic.flink.corpus.SingleOperatorCorpus
  ✓ org.agentic.flink.tool.ToolRegistry


## 2. Web fetch

`WebFetchTool` is a `ToolExecutor` you can wire into an agent, but it's also
directly callable — useful here for the demo.

In [3]:
from agentic_flink import web
from java.util import HashMap

opts = web.options(user_agent='agentic-flink-notebook/1.0', respect_robots=True)
tool = web.fetch_tool(opts)

args = HashMap()
args.put('url', 'https://example.com')

future = tool.execute(args)
result = future.get()  # blocking get on the CompletableFuture

for k in ['url', 'title', 'contentType', 'truncated']:
    print(f'  {k}: {result.get(k)}')
print(f'  text length: {len(str(result.get("text")))} chars')
print(f'  links: {len(result.get("links"))} found')

  url: https://example.com
  title: Example Domain
  contentType: text/html
  truncated: None
  text length: 127 chars
  links: 1 found


## 3. Document extraction

`DocumentExtractor` is what `WebFetchTool` uses internally — Jsoup for HTML,
Tika for everything else. We feed it raw bytes here to show both paths.

In [4]:
DocumentExtractor = jclass('org.agentic.flink.web.DocumentExtractor')
extractor = DocumentExtractor()

html = b'''<html><head><title>Hello</title></head>
<body><h1>Heading</h1><p>Para with <a href="/next">a link</a>.</p></body></html>'''

doc = extractor.extract('https://demo.example/', html, 'text/html')
print('Title :', doc.getTitle())
print('Text  :', doc.getText())
print('Links :', list(doc.getLinks()))

print()
plain = b'Lorem ipsum dolor sit amet, consectetur adipiscing elit.'
doc2 = extractor.extract('https://demo.example/notes.txt', plain, 'text/plain')
print('Tika  :', doc2.getText()[:60], '...')

Title : Hello
Text  : Heading Para with a link.
Links : ['https://demo.example/next']



Tika  : Lorem ipsum dolor sit amet, consectetur adipiscing elit.
 ...


## 4. Chunking

`RecursiveTextChunker` is a LangChain-style splitter: prefers paragraph,
then sentence, then word, then char boundaries. Configurable overlap.

In [5]:
from agentic_flink import ingest

doc_text = (
    'Apache Flink is a stream processing framework. ' * 20 +
    '\n\n' +
    'Agentic Flink builds LLM-powered agents on top. ' * 20
)

chunker = ingest.recursive_chunker(max_chars=200, overlap=30)
chunks = ingest.chunk(chunker, source_id='quickstart-doc', text=doc_text)

print(f'Produced {len(chunks)} chunks')
for c in chunks[:3]:
    print(f'  [{c["position"]:>2}] {len(c["text"]):>3} chars  id={c["id"]}')
print(f'  ... and {len(chunks) - 3} more')
print()
print('First chunk preview:')
print('  ' + chunks[0]['text'][:120] + '...')

Produced 13 chunks
  [ 0] 183 chars  id=quickstart-doc::0
  [ 1] 168 chars  id=quickstart-doc::1
  [ 2] 168 chars  id=quickstart-doc::2
  ... and 10 more

First chunk preview:
  Apache Flink is a stream processing framework Apache Flink is a stream processing framework Apache Flink is a stream pro...


## 5. In-JVM vector memory

Both `FlinkStateVectorMemory` (brute-force) and `FlinkStateHnswVectorMemory`
(HNSW graph) are normally bound inside a Flink operator via `RichFunction.open()`.
Here we drive the lower-level math primitive directly to demonstrate ANN search.

In [6]:
import random, math

random.seed(42)

def random_unit_vec(dim):
    v = [random.gauss(0, 1) for _ in range(dim)]
    norm = math.sqrt(sum(x*x for x in v))
    return [x / norm for x in v]

DIM = 64
vectors = {f'doc-{i}': random_unit_vec(DIM) for i in range(500)}

# Build a planted nearest neighbour for our query so the result is verifiable.
query = random_unit_vec(DIM)
vectors['planted-nn'] = [
    x + 0.05 * random.gauss(0, 1) for x in query
]
# renormalize
norm = math.sqrt(sum(x*x for x in vectors['planted-nn']))
vectors['planted-nn'] = [x / norm for x in vectors['planted-nn']]

print(f'Indexed {len(vectors)} vectors at d={DIM}')

Indexed 501 vectors at d=64


In [7]:
# Vector search inside the framework runs inside a Flink operator (state-backed).
# In a notebook we can't bind a RichFunction, so we exercise the same math in
# Python — the score formula is the contract.

def cosine(a, b):
    dot = sum(x * y for x, y in zip(a, b))
    na = math.sqrt(sum(x * x for x in a))
    nb = math.sqrt(sum(y * y for y in b))
    return dot / (na * nb) if na and nb else 0.0

scored = [(k, cosine(query, v)) for k, v in vectors.items()]
scored.sort(key=lambda x: -x[1])

print('Top-5 by cosine similarity:')
for k, s in scored[:5]:
    marker = '  ← planted' if k == 'planted-nn' else ''
    print(f'  {s:+.4f}  {k}{marker}')

assert scored[0][0] == 'planted-nn', 'planted vector should be the nearest neighbour'
print()
print('✓ planted nearest neighbour recovered at rank 1')

Top-5 by cosine similarity:
  +0.9407  planted-nn  ← planted
  +0.3518  doc-284
  +0.3098  doc-119
  +0.3087  doc-461
  +0.2810  doc-163

✓ planted nearest neighbour recovered at rank 1


In [8]:
# Same payload through HNSW — spec construction only (binding into Flink state
# happens inside a RichFunction.open(), which a notebook can't host).
from agentic_flink import memory

spec_bf = memory.flink_state_brute_force(dimension=DIM, similarity='cosine')
spec_hnsw = memory.flink_state_hnsw(dimension=DIM, similarity='cosine', m=16, beam_width=64)

print('Brute-force spec :', spec_bf.providerName())
print('HNSW spec        :', spec_hnsw.providerName())

Brute-force spec : FlinkStateVectorMemory(brute-force, d=64, sim=COSINE)
HNSW spec        : FlinkStateHnswVectorMemory(d=64, M=16, beam=64, search=50)


## 6. Agent + Python tool (inspect-only — no LLM call yet)

Build an agent end-to-end and inspect what we'd send to the LLM. We don't
invoke it here; section 7 does that conditionally.

In [9]:
from agentic_flink import Agent, ChatSetup, langchain4j_ollama, tool

@tool
def add(a: int, b: int) -> int:
    '''Add two integers.'''
    return a + b

@tool(name='multiply', description='Multiply two integers.')
def mul(a: int, b: int) -> int:
    return a * b

agent = (
    Agent.builder()
        .with_id('calc-bot')
        .with_system_prompt('You are a careful calculator. Use tools for arithmetic.')
        .with_chat_connection(langchain4j_ollama(base_url='http://localhost:11434'))
        .with_chat_setup(ChatSetup(model='qwen2.5:3b', temperature=0.1))
        .with_tools(add, mul)
        .with_max_iterations(4)
        .build()
)

print(agent)
print('  id           :', agent.id)
print('  system_prompt:', agent.system_prompt[:60], '...')
print('  allowed_tools:', sorted(agent.allowed_tools))
print('  max_iter     :', agent.max_iterations)
print('  temperature  :', agent.temperature)

Agent(id='calc-bot', tools=['add', 'multiply'])
  id           : calc-bot
  system_prompt: You are a careful calculator. Use tools for arithmetic. ...
  allowed_tools: ['add', 'multiply']
  max_iter     : 4
  temperature  : 0.1


## 7. Live agent run (skipped if Ollama isn't running)

In [10]:
import urllib.request, urllib.error, json

OLLAMA = 'http://localhost:11434'
MODEL = 'qwen2.5:3b'

def ollama_reachable():
    try:
        with urllib.request.urlopen(f'{OLLAMA}/api/tags', timeout=2) as r:
            return r.status == 200
    except (urllib.error.URLError, ConnectionError, OSError):
        return False

def model_present(name):
    try:
        with urllib.request.urlopen(f'{OLLAMA}/api/tags', timeout=2) as r:
            tags = json.loads(r.read())
            return any(m['name'].startswith(name) for m in tags.get('models', []))
    except Exception:
        return False

if not ollama_reachable():
    print('⚠ Ollama not reachable at', OLLAMA, '— skipping live run.')
    print('  Start with: podman run -d --name ollama -p 11434:11434 ollama/ollama')
elif not model_present(MODEL.split(':')[0]):
    print(f'⚠ Model {MODEL!r} not pulled — skipping live run.')
    print(f'  Pull with: podman exec ollama ollama pull {MODEL}')
else:
    # Drive the chat connection directly to prove end-to-end reachability
    # without spinning up a Flink job for one query.
    from agentic_flink.llm import chat, langchain4j_ollama, ChatSetup, ChatMessage
    conn = langchain4j_ollama(base_url=OLLAMA)
    setup = ChatSetup(model=MODEL, temperature=0.1)
    reply = chat(conn, setup, [ChatMessage.user('Say the single word: ready')])
    print('LLM replied :', reply)


⚠ Ollama not reachable at http://localhost:11434 — skipping live run.
  Start with: podman run -d --name ollama -p 11434:11434 ollama/ollama


## Done

If you got to the bottom without errors:
- The JVM started and the framework jar resolved
- A real web page was fetched and parsed
- Document extraction worked for both HTML (Jsoup) and plain text (Tika)
- Recursive chunking produced overlapping chunks
- Vector similarity search found the planted nearest neighbour
- Memory specs for brute-force and HNSW constructed cleanly
- An `Agent` built with two Python tools and inspected
- (Optional) A live chat call returned a token from Ollama

Next: `notebooks/02_rag_pipeline.ipynb` (TBD) walks through ingest → corpus → retrieve
with a real corpus of pages.